# Information Theory Foundations Lab

## Surprise, entropy, cross-entropy, KL, mutual information, coding, and ELBO

This guided lab accompanies the [information-theory prerequisite track](https://github.com/jjames/llm-wiki/tree/main/lessons/prerequisites/04-information-theory).

**Recommended order:** prerequisite lessons → this notebook → Notebook 16 mastery  
**Time:** 80–110 minutes  
**Dependencies:** NumPy, Matplotlib, and Python's standard library

### Learning goals

You will interpret information quantities as expected code lengths, verify the cross-entropy decomposition, calculate mutual information from joint tables, observe data processing, construct a Huffman code, and derive an ELBO gap in a fully enumerable latent model.


In [ ]:
import heapq

import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=6, suppress=True)


## 1. Self-information and entropy

An event with probability $p$ carries self-information

$$I(x)=-\log_2p(x).$$

Independent event information adds because probabilities multiply and logarithms turn products into sums. Entropy is expected surprise:

$$H(X)=-\sum_xp(x)\log_2p(x).$$

Among $k$ outcomes, the uniform distribution has maximum entropy $\log_2k$.


In [ ]:
def entropy(probabilities):
    p = np.asarray(probabilities, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

distributions = {
    "certain": np.array([1.0, 0.0, 0.0, 0.0]),
    "skewed": np.array([0.7, 0.1, 0.1, 0.1]),
    "uniform": np.full(4, 0.25),
}
entropies = {name: entropy(p) for name, p in distributions.items()}
assert entropies["certain"] == 0.0
assert entropies["uniform"] == 2.0
assert entropies["certain"] < entropies["skewed"] < entropies["uniform"]
np.testing.assert_allclose(-np.log2(0.25 * 0.125), -np.log2(0.25) - np.log2(0.125))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(entropies.keys(), entropies.values(), color=["C0", "C1", "C2"])
ax.set(ylabel="bits", title="Entropy measures expected surprise, not outcome count alone")
ax.grid(alpha=0.2, axis="y")
plt.show()
print(entropies)


## 2. Cross-entropy and KL divergence

If data comes from $p$ but we encode or predict with $q$, expected code length is cross-entropy

$$H(p,q)=-\sum_xp(x)\log_2q(x).$$

It decomposes as

$$H(p,q)=H(p)+D_{KL}(p\Vert q).$$

KL is nonnegative and zero only when the distributions match (almost everywhere). It is asymmetric because using $q$ to encode $p$ is a directed mistake.


In [ ]:
def cross_entropy(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    return -np.sum(p[p > 0] * np.log2(q[p > 0]))

def kl_divergence(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    mask = p > 0
    return np.sum(p[mask] * np.log2(p[mask] / q[mask]))

p = np.array([0.5, 0.3, 0.2])
q = np.array([0.4, 0.4, 0.2])
np.testing.assert_allclose(cross_entropy(p, q), entropy(p) + kl_divergence(p, q))
assert kl_divergence(p, q) >= 0
assert not np.isclose(kl_divergence(p, q), kl_divergence(q, p))

logits = np.array([2.0, 0.5, -1.0])
probabilities = np.exp(logits - logits.max()); probabilities /= probabilities.sum()
one_hot_target = np.array([1.0, 0.0, 0.0])
nll = -np.log2(probabilities[0])
np.testing.assert_allclose(cross_entropy(one_hot_target, probabilities), nll)
print(f"H(p)={entropy(p):.4f}, KL(p||q)={kl_divergence(p,q):.4f}, H(p,q)={cross_entropy(p,q):.4f} bits")


## 3. Mutual information from a joint distribution

Mutual information measures how much observing one variable reduces uncertainty about another:

$$I(X;Y)=D_{KL}(p(x,y)\Vert p(x)p(y))=H(X)-H(X\mid Y).$$

It is zero for independence and symmetric in $X,Y$. Unlike correlation, it can detect nonlinear and non-numeric dependence.


In [ ]:
def mutual_information(joint):
    joint = np.asarray(joint, dtype=float)
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    independent_reference = px @ py
    mask = joint > 0
    return np.sum(joint[mask] * np.log2(joint[mask] / independent_reference[mask]))

independent = np.full((2, 2), 0.25)
perfect_copy = np.array([[0.5, 0.0], [0.0, 0.5]])
noisy_copy = np.array([[0.45, 0.05], [0.05, 0.45]])
assert abs(mutual_information(independent)) < 1e-12
np.testing.assert_allclose(mutual_information(perfect_copy), 1.0)
assert 0 < mutual_information(noisy_copy) < 1
np.testing.assert_allclose(mutual_information(noisy_copy), mutual_information(noisy_copy.T))
print(f"MI independent={mutual_information(independent):.3f}, noisy copy={mutual_information(noisy_copy):.3f}, perfect={mutual_information(perfect_copy):.3f} bits")


## 4. Data processing inequality

If $X\to Y\to Z$ is a Markov chain, processing $Y$ without fresh access to $X$ cannot create information about $X$:

$$I(X;Z)\le I(X;Y).$$

Equality is possible for lossless processing; strict inequality appears when the second channel discards signal.


In [ ]:
def binary_channel_joint(flip_probability):
    # Uniform binary source through a binary symmetric channel.
    return np.array([
        [0.5 * (1 - flip_probability), 0.5 * flip_probability],
        [0.5 * flip_probability, 0.5 * (1 - flip_probability)],
    ])

first_flip = 0.1
second_flips = np.linspace(0, 0.5, 51)
effective_flips = first_flip * (1 - second_flips) + (1 - first_flip) * second_flips
i_xy = mutual_information(binary_channel_joint(first_flip))
i_xz = np.array([mutual_information(binary_channel_joint(p)) for p in effective_flips])
assert np.all(i_xz <= i_xy + 1e-12)
np.testing.assert_allclose(i_xz[0], i_xy)
assert abs(i_xz[-1]) < 1e-12

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(second_flips, i_xz, label="I(X;Z)")
ax.axhline(i_xy, color="C1", linestyle="--", label="I(X;Y)")
ax.set(xlabel="second-channel flip probability", ylabel="bits", title="Further noisy processing cannot add source information")
ax.legend(); ax.grid(alpha=0.25)
plt.show()


## 5. Prefix codes and the entropy bound

A binary prefix code lets a decoder identify symbol boundaries without separators. Huffman coding greedily combines the least probable symbols and minimizes expected length among binary prefix codes with integer lengths.

For an optimal binary prefix code,

$$H(X)\le E[L]<H(X)+1.$$


In [ ]:
def huffman_lengths(probabilities):
    heap = [(float(probability), [index]) for index, probability in enumerate(probabilities)]
    heapq.heapify(heap)
    lengths = np.zeros(len(probabilities), dtype=int)
    while len(heap) > 1:
        p_left, left_symbols = heapq.heappop(heap)
        p_right, right_symbols = heapq.heappop(heap)
        for symbol in left_symbols + right_symbols:
            lengths[symbol] += 1
        heapq.heappush(heap, (p_left + p_right, left_symbols + right_symbols))
    return lengths

probabilities = np.array([0.40, 0.25, 0.20, 0.10, 0.05])
lengths = huffman_lengths(probabilities)
expected_length = probabilities @ lengths
source_entropy = entropy(probabilities)
kraft_sum = np.sum(2.0 ** (-lengths))
assert source_entropy <= expected_length < source_entropy + 1
assert kraft_sum <= 1 + 1e-12
print("probabilities:", probabilities)
print("Huffman lengths:", lengths)
print(f"entropy={source_entropy:.3f} bits; expected code length={expected_length:.3f} bits; Kraft sum={kraft_sum:.3f}")


## 6. The ELBO in an enumerable latent model

For latent variable $Z$ and observation $x$,

$$\log p(x)=\operatorname{ELBO}(q)+D_{KL}(q(z)\Vert p(z\mid x)).$$

The evidence lower bound is lower because KL is nonnegative. It becomes tight when the approximate posterior $q$ equals the exact posterior. This is both an optimization objective and a coding statement.


In [ ]:
# Binary latent state; observe x=1.
prior = np.array([0.7, 0.3])
likelihood_x1 = np.array([0.2, 0.9])
joint_x1 = prior * likelihood_x1
evidence = joint_x1.sum()
posterior = joint_x1 / evidence

q1_grid = np.linspace(0.001, 0.999, 500)
elbos = []
gaps = []
for q1 in q1_grid:
    q = np.array([1 - q1, q1])
    elbo = np.sum(q * (np.log(joint_x1) - np.log(q)))
    gap = np.sum(q * (np.log(q) - np.log(posterior)))
    elbos.append(elbo); gaps.append(gap)
elbos = np.array(elbos); gaps = np.array(gaps)
np.testing.assert_allclose(np.log(evidence), elbos + gaps, atol=1e-12)
assert np.all(elbos <= np.log(evidence) + 1e-12)
best_q1 = q1_grid[np.argmax(elbos)]
assert abs(best_q1 - posterior[1]) < 0.003

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q1_grid, elbos, label="ELBO(q)")
ax.axhline(np.log(evidence), color="C1", linestyle="--", label="log evidence")
ax.axvline(posterior[1], color="C2", linestyle=":", label="exact posterior q(1)")
ax.set(xlabel="q(z=1)", ylabel="nats", title="The ELBO becomes tight at the posterior")
ax.legend(); ax.grid(alpha=0.25)
plt.show()
print("posterior:", posterior, "best grid q(z=1):", best_q1)


## Cumulative mastery check

1. Why does an unlikely event carry more information than a likely one?
2. Explain $H(p,q)=H(p)+KL(p\Vert q)$ as a coding penalty.
3. Construct dependent variables with zero linear correlation but positive mutual information.
4. When can deterministic processing preserve mutual information exactly?
5. Why can Huffman lengths not generally equal ideal lengths $-\log_2p(x)$?
6. In the ELBO identity, which distribution makes the bound tight and why?

**Next:** Notebook 16 combines entropy, mutual information, data processing, variational inference, ELBO, and bits-back coding in a less guided mastery lab.
